<a href="https://colab.research.google.com/github/SMLFRD20/PROGRAMACION_PARALELA_DISTRIBUIDA/blob/main/Hadoop_MapReduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import os


!apt-get update -qq
!apt-get install openjdk-8-jdk-headless -qq > /dev/null


if not os.path.exists("hadoop-3.3.6.tar.gz"):
    !wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
if not os.path.exists("/usr/local/hadoop-3.3.6"):
    !tar -xzf hadoop-3.3.6.tar.gz
    !cp -r hadoop-3.3.6 /usr/local/

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/usr/local/hadoop-3.3.6"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["HADOOP_HOME"] + "/sbin:" + os.environ["PATH"]

conf_dir = os.path.join(os.environ["HADOOP_HOME"], "etc/hadoop")
!rm -rf $conf_dir
!mkdir -p $conf_dir

with open(os.path.join(conf_dir, "core-site.xml"), "w") as f:
    f.write('<configuration><property><name>fs.defaultFS</name><value>file:///</value></property></configuration>')

with open(os.path.join(conf_dir, "hadoop-env.sh"), "w") as f:
    f.write(f'export JAVA_HOME={os.environ["JAVA_HOME"]}')

print("Hadoop RECONFIGURADO en modo LOCAL PURO. Se ha desactivado cualquier intento de usar HDFS.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Hadoop RECONFIGURADO en modo LOCAL PURO. Se ha desactivado cualquier intento de usar HDFS.


In [6]:
%%writefile mapper.py
import sys
import re

def main():
    for line in sys.stdin:
        line = line.strip().lower()
        line = re.sub(r'[^\w\s]', '', line)
        words = line.split()
        for word in words:
            if word:
                print(f"{word}\t1")

if __name__ == "__main__":
    main()

Overwriting mapper.py


In [7]:
%%writefile reducer.py
import sys

def main():
    current_word = None
    current_count = 0
    word = None

    for line in sys.stdin:
        line = line.strip()
        try:
            word, count = line.split('\t', 1)
            count = int(count)
        except ValueError:
            continue

        if current_word == word:
            current_count += count
        else:
            if current_word:
                print(f"{current_word}\t{current_count}")
            current_word = word
            current_count = count

    if current_word == word:
        print(f"{current_word}\t{current_count}")

if __name__ == "__main__":
    main()

Overwriting reducer.py


In [8]:
!chmod +x mapper.py reducer.py
!mkdir -p datos_locales

In [9]:
%%writefile datos_locales/documento1.txt
Hadoop es un framework de codigo abierto para almacenar y procesar grandes volumenes de datos.
MapReduce es un modelo de programacion nativo de Hadoop para el procesamiento distribuido.
Big Data requiere herramientas eficientes como Hadoop y Spark.

Writing datos_locales/documento1.txt


In [10]:
%%writefile datos_locales/documento2.txt
El procesamiento distribuido divide las tareas en nodos del cluster.
MapReduce tiene dos etapas principales: la etapa de Map y la etapa de Reduce.
Hadoop utiliza HDFS para el almacenamiento confiable de datos distribuidos.

Writing datos_locales/documento2.txt


In [29]:
!mkdir -p input_files
!cp /content/datos_locales/*.txt /content/input_files/
print("Archivos listos en el directorio local 'input_files' para procesamiento.")
!ls /content/input_files/

Archivos listos en el directorio local 'input_files' para procesamiento.
documento1.txt	documento2.txt


In [34]:
JAR_PATH="/usr/local/hadoop-3.3.6/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar"

!rm -rf /content/output /content/output_2r

print("--- EJECUTANDO WORDCOUNT (1 REDUCER) ---")
!hadoop jar $JAR_PATH \
    -D fs.defaultFS=file:/// \
    -files /content/mapper.py,/content/reducer.py \
    -mapper "python3 mapper.py" \
    -reducer "python3 reducer.py" \
    -input /content/input_files/*.txt \
    -output /content/output

if os.path.exists("/content/output/part-00000"):
    print("\n--- RESULTADOS ---")
    !cat /content/output/part-00000 | head -n 10
else:
    print("\nError: El archivo de salida no se generó.")

--- EJECUTANDO WORDCOUNT (1 REDUCER) ---
2026-07-09 01:07:13,173 WARN  [main] impl.MetricsConfig (MetricsConfig.java:loadFirst(136)) - Cannot locate configuration: tried hadoop-metrics2-jobtracker.properties,hadoop-metrics2.properties
2026-07-09 01:07:13,276 INFO  [main] impl.MetricsSystemImpl (MetricsSystemImpl.java:startTimer(378)) - Scheduled Metric snapshot period at 10 second(s).
2026-07-09 01:07:13,276 INFO  [main] impl.MetricsSystemImpl (MetricsSystemImpl.java:start(191)) - JobTracker metrics system started
2026-07-09 01:07:13,297 WARN  [main] impl.MetricsSystemImpl (MetricsSystemImpl.java:init(151)) - JobTracker metrics system already initialized!
2026-07-09 01:07:13,655 INFO  [main] mapred.FileInputFormat (FileInputFormat.java:listStatus(266)) - Total input files to process : 2
2026-07-09 01:07:13,684 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java:submitJobInternal(202)) - number of splits:2
2026-07-09 01:07:14,096 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java

In [36]:
print("--- RESULTADOS FINALES (LOCAL) ---")
!cat /content/output/part-00000 | head -n 20

--- RESULTADOS FINALES (LOCAL) ---
abierto	1
almacenamiento	1
almacenar	1
big	1
cluster	1
codigo	1
como	1
confiable	1
data	1
datos	2
de	7
del	1
distribuido	2
distribuidos	1
divide	1
dos	1
eficientes	1
el	3
en	1
es	2


In [35]:
JAR_PATH="/usr/local/hadoop-3.3.6/share/hadoop/tools/lib/hadoop-streaming-3.3.6.jar"

print("=== EJECUCIÓN CON 2 REDUCERS ===")
!rm -rf /content/output_2r

!time hadoop jar $JAR_PATH \
    -D fs.defaultFS=file:/// \
    -files /content/mapper.py,/content/reducer.py \
    -mapper "python3 mapper.py" \
    -reducer "python3 reducer.py" \
    -numReduceTasks 2 \
    -input /content/input_files/*.txt \
    -output /content/output_2r

print("\nResultados guardados en /content/output_2r")
!ls /content/output_2r

=== EJECUCIÓN CON 2 REDUCERS ===
2026-07-09 01:07:39,368 WARN  [main] impl.MetricsConfig (MetricsConfig.java:loadFirst(136)) - Cannot locate configuration: tried hadoop-metrics2-jobtracker.properties,hadoop-metrics2.properties
2026-07-09 01:07:39,475 INFO  [main] impl.MetricsSystemImpl (MetricsSystemImpl.java:startTimer(378)) - Scheduled Metric snapshot period at 10 second(s).
2026-07-09 01:07:39,475 INFO  [main] impl.MetricsSystemImpl (MetricsSystemImpl.java:start(191)) - JobTracker metrics system started
2026-07-09 01:07:39,499 WARN  [main] impl.MetricsSystemImpl (MetricsSystemImpl.java:init(151)) - JobTracker metrics system already initialized!
2026-07-09 01:07:39,844 INFO  [main] mapred.FileInputFormat (FileInputFormat.java:listStatus(266)) - Total input files to process : 2
2026-07-09 01:07:39,872 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java:submitJobInternal(202)) - number of splits:2
2026-07-09 01:07:40,114 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java:printTo